*Auto-generated from a Mathcad worksheet by mcad2py.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import pint

from mcad2py.runtime import augment, mround, nth_root, power, disp, elementwise, mc_min, mc_max, matrix, matelem, matmul, col, arange, vectorize, transpose, matcol, total, vec_set, solve_block, sample, plot_axis
ureg = pint.UnitRegistry()

LT91

1 Introduction

...

2 Properties

Column dimensions

In [ ]:
w_c = 1.3 * ureg.m

In [ ]:
l_c = 1.3 * ureg.m

Foundation thickness:

In [ ]:
t = 1.0 * ureg.m

In [ ]:
Contour = 1 / 2 * matmul(matrix(5, 2, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1), matrix(2, 2, w_c, 0, 0, l_c))

Foundation level:

In [ ]:
FUK = 16.5 * ureg.m

Cover layer

In [ ]:
cov = 45 * ureg.mm

In [ ]:
z = 19.42 * ureg.m

Top of column:

In [ ]:
h_c = z - (FUK + t)
h_c

Height of column:

2.1 Materials

Concrete properties:

In [ ]:
f_ck = 35 * ureg.MPa

In [ ]:
f_cd = f_ck / 1.5
disp(f_cd, ureg.MPa)

In [ ]:
E_c = 30 * ureg.GPa

Strain limits

In [ ]:
epsilon_c2 = -2 * 10**-3

In [ ]:
epsilon_cu = -3.5 * 10**-3

Steel properties:

In [ ]:
f_yk = 500 * ureg.MPa

In [ ]:
f_yd = f_yk / 1.15

In [ ]:
E_s = 200 * ureg.GPa

In [ ]:
alpha = E_s / E_c
disp(alpha)

Strain limits

In [ ]:
epsilon_yd = f_yd / E_s
disp(epsilon_yd)

In [ ]:
epsilon_ud = 9 / 100

Stress-strain functions:

In [ ]:
def sigma_c(e):
    if e > 0:
        return 0 * ureg.MPa
    elif e > epsilon_c2:
        return -f_cd * (1 - (1 - e / epsilon_c2)**2)
    return -f_cd
sigma_c = elementwise(sigma_c)

In [ ]:
sigma_s = lambda e: mc_min(f_yd, mc_max(-f_yd, E_s * e))
sigma_s = elementwise(sigma_s)

2.1 Reinforcement

Stirrup:

In [ ]:
ø_w = 12 * ureg.mm

In [ ]:
s_w = 400 * ureg.mm

Vertical reinforcement:

In [ ]:
ø = 20 * ureg.mm

Target spacing (used to calcualte number of bars)

In [ ]:
s_x = 300 * ureg.mm

In [ ]:
s_y = 300 * ureg.mm

In [ ]:
n_s = col(mround(l_c / s_x), mround(w_c / s_y), mround(l_c / s_x), mround(w_c / s_y)) - 2
disp(n_s)

In [ ]:
stirrup = Contour + matrix(5, 2, cov + ø_w / 2, cov + ø_w / 2, -cov - ø_w / 2, -cov - ø_w / 2, cov + ø_w / 2, cov + ø_w / 2, -cov - ø_w / 2, -cov - ø_w / 2, cov + ø_w / 2, cov + ø_w / 2)
disp(stirrup)

In [ ]:
s = vectorize(col(l_c - 2 * cov - 2 * ø_w, w_c - 2 * cov - 2 * ø_w, l_c - 2 * cov - 2 * ø_w, w_c - 2 * cov - 2 * ø_w) * (1 / (n_s + 1)))
disp(s, ureg.mm)

In [ ]:
def _X_s_Y_s_n():
    X = None
    Y = None
    j = 0
    for i in arange(0, n_s[0], 1):
        X = vec_set(X, j, -l_c / 2 + cov + ø_w + ø / 2 + i * s[0])
        Y = vec_set(Y, j, w_c / 2 - cov - ø_w - ø / 2)
        j = j + 1
    for i in arange(0, n_s[1], 1):
        X = vec_set(X, j, l_c / 2 - cov - ø_w - ø / 2)
        Y = vec_set(Y, j, w_c / 2 - cov - ø_w - s[1] * i - ø / 2)
        j = j + 1
    for i in arange(0, n_s[2], 1):
        X = vec_set(X, j, l_c / 2 - cov - ø_w - s[2] * i - ø / 2)
        Y = vec_set(Y, j, -w_c / 2 + cov + ø_w + ø / 2)
        j = j + 1
    for i in arange(0, n_s[3], 1):
        X = vec_set(X, j, -l_c / 2 + cov + ø_w + ø / 2)
        Y = vec_set(Y, j, -w_c / 2 + cov + ø_w + s[3] * i + ø / 2)
        j = j + 1
    return col(X, Y, j)
X_s, Y_s, n = tuple(_X_s_Y_s_n())

Actual spacing on each face:

In [ ]:
disp((s), ureg.mm)

Total number of vertical bars:

In [ ]:
n

2.3 Gross section properties

Area:

In [ ]:
A_c = w_c * l_c
A_c

In [ ]:
def _A_s():
    A = None
    for i in arange(0, n - 1, 1):
        A = vec_set(A, i, ø**2 * (math.pi / 4))
    return A
A_s = _A_s()

Moment of inertia:

In [ ]:
Iy = 1 / 12 * w_c * l_c**3

In [ ]:
Ix = 1 / 12 * w_c**3 * l_c

2.3 Cross section

The cross section is divided into 10x10 fibers to solve the biaxial problem

Division:

In [ ]:
n_x = 10

In [ ]:
n_y = 10

Size of each fiber:

In [ ]:
dy = w_c / n_y
disp(dy, ureg.mm)

In [ ]:
dx = l_c / n_x
disp(dx, ureg.mm)

In [ ]:
A_ci = dx * dy
A_ci

In [ ]:
def _X_c_Y_c():
    Y = None
    X = None
    for i in arange(0, n_x - 1, 1):
        for j in arange(0, n_y - 1, 1):
            Y = vec_set(Y, i + j * n_x, -w_c / 2 + (j + 0.5) * dy)
            X = vec_set(X, i + j * n_x, -l_c / 2 + (i + 0.5) * dx)
    return col(X, Y)
X_c, Y_c = tuple(_X_c_Y_c())

In [ ]:
_fig, _ax = plt.subplots()
_ax.plot(plot_axis(matcol(Contour, 0), ureg.mm), plot_axis(matcol(Contour, 1), ureg.mm), label='matcol(Contour, 0)', color='#00008B')
_ax.plot(plot_axis(matcol(stirrup, 0), ureg.mm), plot_axis(matcol(stirrup, 1), ureg.mm), label='matcol(stirrup, 0)', color='#932329')
_ax.plot(plot_axis(X_s, ureg.mm), plot_axis(Y_s, ureg.mm), label='X_s', color='#932329')
_ax.plot(plot_axis(X_c, ureg.mm), plot_axis(Y_c, ureg.mm), label='X_c', color='#A1A3A6')
_ax.axhline(0, color='0.6', linewidth=0.8)
_ax.axvline(0, color='0.6', linewidth=0.8)
_ax.grid(True, alpha=0.3)
_ax.set_xlabel('(mm)')
_ax.set_ylabel('(mm)')
_ax.legend()
plt.show()

Center of each sub-division fiber shown as grey marker:

3 Forces

In [ ]:
LS = col('ULS', 'ULS', 'ULS', 'ULS', 'ALS', 'ALS', 'ULS', 'ULS', 'ULS', 'ULS', 'ULS', 'ULS')

In [ ]:
ID = col('LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91', 'LT91')

In [ ]:
Side = col('North', 'South', 'North', 'South', 'North', 'South', 'North', 'South', 'North', 'South', 'North', 'South')

In [ ]:
Case = col('After Launching', 'After Launching', 'After Launching', 'After Launching', 'After Launching', 'After Launching', 'During Launching', 'During Launching', 'Park case 2', 'Park case 2', 'Park case 2', 'Park case 2')

In [ ]:
WindDir = col('NorthToSouth', 'NorthToSouth', 'SouthToNorth', 'SouthToNorth', '-', '-', '-', '-', 'NorthToSouth', 'NorthToSouth', 'SouthToNorth', 'SouthToNorth')

In [ ]:
Fz = col(1819, 1853, 1850, 1822, 2669, 3036, 8041, 8333, 3945, 4847, 4762, 4030) * ureg.kN

In [ ]:
Fy = col(637, 637, 637, 637, 0, 0, 226, 226, 256, 256, 256, 256) * ureg.kN

In [ ]:
Fx = col(0, 0, 0, 0, 0, 0, 776, 805, 0, 0, 0, 0) * ureg.kN

Weight of column:

In [ ]:
G_c = w_c * l_c * h_c * (25 * (ureg.kN / ureg.m**3))
disp(G_c, ureg.kN)

3.1 Second order effects and imperfections

Effective height (Assumption based on cantilever with elastic rotation stiffness)

In [ ]:
l_0 = h_c * 3

Slenderness:

In [ ]:
I_c = mc_min(Ix, Iy)

In [ ]:
i = nth_root(I_c / A_c, 2)
disp(i)

In [ ]:
lambda_ = l_0 / i
disp(lambda_)

Limiting Slenderness:

In [ ]:
A = 0.7

In [ ]:
B = 1.1

In [ ]:
C = 0.7

In [ ]:
n_0 = mc_max(Fz) / (A_c * f_cd)
disp(n_0)

In [ ]:
lambda__lim = 20 * A * B * C / nth_root(n_0, 2)
disp(lambda__lim)

In [ ]:
print('lambda_ < lambda__lim', lambda_ < lambda__lim, 'Effects can be neglected!')

Imperfections:

In [ ]:
theta_0 = 1 / 200

In [ ]:
alpha_m = 1

In [ ]:
alpha_h = mc_min(2 / nth_root(h_c * (1 / ureg.m), 2), 1)
disp(alpha_h)

In [ ]:
theta_i = theta_0 * alpha_h * alpha_m
theta_i

In [ ]:
e_i = theta_i * l_0 / 2
disp(e_i, ureg.mm)

Placement tolerance for temporary bearings (assumption):

In [ ]:
e_tol = 50 * ureg.mm

3.2 Design forces at base

Normal force incl self weight:

Bending moment incl imperfections and tolerance

In [ ]:
N = -Fz - G_c
disp(N, ureg.kN)

In [ ]:
M_x = Fy * h_c - (e_i + e_tol) * N
disp(M_x, ureg.kN)

In [ ]:
M_y = Fx * h_c - (e_i + e_tol) * N
disp(M_y, ureg.kN)

Shear force:

In [ ]:
V_x = Fx

In [ ]:
V_y = Fy

4 Biaxial problem

Forces in each bar and concrete fiber as a function of strain and section curvature:

In [ ]:
F_si = lambda e, kx, ky: vectorize(A_s * sigma_s(e + kx * X_s + ky * Y_s))

In [ ]:
F_ci = lambda e, kx, ky: vectorize(A_ci * sigma_c(e + kx * X_c + ky * Y_c))

Internal forces in section for given strain parameters:

In [ ]:
N_int = lambda e, kx, ky: total(F_ci(e, kx, ky)) + total(F_si(e, kx, ky))

In [ ]:
M_x_int = lambda e, kx, ky: total(vectorize(F_ci(e, kx, ky) * Y_c)) + total(vectorize(F_si(e, kx, ky) * Y_s))

In [ ]:
M_y_int = lambda e, kx, ky: total(vectorize(F_ci(e, kx, ky) * X_c)) + total(vectorize(F_si(e, kx, ky) * X_s))

Solve equilibrium to find strain profile:

In [ ]:
def solve_strain(N, Mx, My):
    e = N / (A_c * E_c)
    kx = My / (Iy * E_c)
    ky = Mx / (Ix * E_c)
    def _residuals_e_kx_ky(_x):
        e, kx, ky = _x
        return [
            N_int(e, kx, ky) - (N),
            M_x_int(e, kx, ky) - (-Mx),
            M_y_int(e, kx, ky) - (-My),
        ]
    return solve_block(_residuals_e_kx_ky, [e, kx, ky])

In [ ]:
def ones(n):
    A = None
    for i in arange(0, n - 1, 1):
        A = vec_set(A, i, 1)
    return A
ones = elementwise(ones)

Concrete corner strains:

In [ ]:
epsilon_c = lambda e, kx, ky: matmul(matrix(4, 3, 1, 1, 1, 1, -l_c / 2, -l_c / 2, l_c / 2, l_c / 2, -w_c / 2, w_c / 2, -w_c / 2, w_c / 2), col(e, kx, ky))

Rebar strain:

In [ ]:
epsilon_s = lambda e, kx, ky: matmul(augment(ones(n), X_s, Y_s), col(e, kx, ky))

Utilization functions:

In [ ]:
UR_c = lambda epsilon_c: mc_min(epsilon_c) / epsilon_cu

In [ ]:
UR_s = lambda epsilon_s: mc_max(epsilon_s) / epsilon_ud

4.1 Loop through all load cases

All load cases are looped through - the equilibrium strain profile is calculated and saved in the E matrix, and the resulting utilizations are found.The load cases with maximum tensile and compressive utlizations are saved and shown below

In [ ]:
def _UR_c_max_i_c_UR_s_max_i_s_ERR_E():
    E = None
    i_c = -1
    UR_c_max = 0
    i_s = -1
    UR_s_max = 0
    for j in arange(0, len(Case) - 1, 1):
        try:
            e = solve_strain(N[j], M_x[j], M_y[j])
            E = vec_set(E, j, e)
            ec = epsilon_c(e[0], e[1], e[2])
            es = epsilon_s(e[0], e[1], e[2])
            UR_ci = UR_c(ec)
            UR_si = UR_s(es)
            if UR_ci > UR_c_max:
                UR_c_max = UR_ci
                i_c = j
            if UR_si > UR_s_max:
                UR_s_max = UR_si
                i_s = j
        except Exception:
            return transpose(col(9.99, j, 9.99, j, 1, E))
    return transpose(col(UR_c_max, i_c, UR_s_max, i_s, 0, E))
UR_c_max, i_c, UR_s_max, i_s, ERR, E = tuple(_UR_c_max_i_c_UR_s_max_i_s_ERR_E())
[UR_c_max, i_c, UR_s_max, i_s, ERR, E]

In [ ]:
print('ERR == 0', ERR == 0, 'Solved without errors')

In [ ]:
print('mc_max(UR_c_max, UR_s_max) < 1', mc_max(UR_c_max, UR_s_max) < 1, 'All loadcases pass!')

4.2 Critical load cases

4.2.1 Maximum compression case:

neural axis (found by solving strain = 0 on each boundary)

Critical case nr:

In [ ]:
j = i_c
j

In [ ]:
def Neutral(e, kx, ky):
    Ans = None
    y = -e / ky + l_c / 2 * (kx / ky)
    j = 0
    if y >= -w_c / 2 and y <= w_c / 2:
        Ans = vec_set(Ans, (j, 0), -l_c / 2)
        Ans = vec_set(Ans, (j, 1), y)
        j = j + 1
    y = -e / ky - l_c / 2 * (kx / ky)
    if y > -w_c / 2 and y < w_c / 2:
        Ans = vec_set(Ans, (j, 0), l_c / 2)
        Ans = vec_set(Ans, (j, 1), y)
        j = j + 1
    x = -e / kx + w_c / 2 * (ky / kx)
    if x >= -l_c / 2 and x <= l_c / 2:
        Ans = vec_set(Ans, (j, 0), x)
        Ans = vec_set(Ans, (j, 1), -w_c / 2)
        j = j + 1
    x = -e / kx - w_c / 2 * (ky / kx)
    if x > -l_c / 2 and x < l_c / 2:
        Ans = vec_set(Ans, (j, 0), x)
        Ans = vec_set(Ans, (j, 1), w_c / 2)
        j = j + 1
    return Ans

Case info:

In [ ]:
LS[j]

In [ ]:
ID[j]

In [ ]:
Side[j]

In [ ]:
Case[j]

In [ ]:
WindDir[j]

Section forces (incl imperfections):

In [ ]:
disp((N[j]), ureg.kN)

In [ ]:
disp((M_x[j]), ureg.kN * ureg.m)

In [ ]:
disp((M_y[j]), ureg.kN * ureg.m)

Resulting strain parameters:

In [ ]:
e, kx, ky = tuple(E[j])
[e, kx, ky]

In [ ]:
NA = Neutral(e, kx, ky)
NA

find location of resultants:

In [ ]:
t_only = lambda x: mc_max(0, x)
t_only = elementwise(t_only)

In [ ]:
c_only = lambda x: mc_min(0, x)
c_only = elementwise(c_only)

Concrete strains/stress at corners:

In [ ]:
e_c = epsilon_c(e, kx, ky)
e_c

In [ ]:
disp((vectorize(sigma_c(e_c))), ureg.MPa)

In [ ]:
T = lambda e, kx, ky: total(vectorize(t_only(F_si(e, kx, ky))))

In [ ]:
C = lambda e, kx, ky: total(vectorize(c_only(F_ci(e, kx, ky)))) + total(vectorize(c_only(F_si(e, kx, ky))))

In [ ]:
disp((mc_min(vectorize(sigma_c(e_c)))), ureg.MPa)

In [ ]:
CG = lambda e, kx, ky: matrix(2, 2, total(vectorize(t_only(F_si(e, kx, ky)) * X_s)) / T(e, kx, ky), (total(vectorize(c_only(F_ci(e, kx, ky)) * X_c)) + total(vectorize(c_only(F_si(e, kx, ky)) * X_s))) / C(e, kx, ky), total(vectorize(t_only(F_si(e, kx, ky)) * Y_s)) / T(e, kx, ky), (total(vectorize(c_only(F_ci(e, kx, ky)) * Y_c)) + total(vectorize(c_only(F_si(e, kx, ky)) * Y_s))) / C(e, kx, ky))

Strain/stress in rebar:

In [ ]:
e_s = epsilon_s(e, kx, ky)
e_s

In [ ]:
vectorize(sigma_s(e_s))

In [ ]:
CGi = CG(e, kx, ky)
CGi

In [ ]:
disp((mc_max(vectorize(sigma_s(e_s)))), ureg.MPa)

In [ ]:
def z(e, kx, ky):
    X = CG(e, kx, ky)
    return nth_root((matelem(X, 1, 0) - matelem(X, 0, 0))**2 + (matelem(X, 1, 1) - matelem(X, 0, 1))**2, 2)

Utilizations (in terms of strain limits):

In [ ]:
UR_c(e_c)

In [ ]:
UR_s(e_s)

Cross section w/ resulting neutral axis (green dash) and location of Compression/Tension resultants (purple stars, lever arm shown dash between resultants):

In [ ]:
A_stx = lambda e, kx, ky: t_only(F_si(e, kx, ky)) / t_only(vectorize(sigma_s(epsilon_s(e, kx, ky))))

In [ ]:
disp((z(e, kx, ky)), ureg.mm)

In [ ]:
_fig, _ax = plt.subplots()
_ax.plot(plot_axis(matcol(Contour, 0), ureg.mm), plot_axis(matcol(Contour, 1), ureg.mm), label='matcol(Contour, 0)', color='#00008B')
_ax.plot(plot_axis(X_s, ureg.mm), plot_axis(Y_s, ureg.mm), label='X_s', color='#932329')
_ax.plot(plot_axis(matcol(NA, 0), ureg.mm), plot_axis(matcol(NA, 1), ureg.mm), label='matcol(NA, 0)', color='#068149')
_ax.plot(plot_axis(matcol(CGi, 0), ureg.mm), plot_axis(matcol(CGi, 1), ureg.mm), label='matcol(CGi, 0)', color='#662D91')
_ax.axhline(0, color='0.6', linewidth=0.8)
_ax.axvline(0, color='0.6', linewidth=0.8)
_ax.grid(True, alpha=0.3)
_ax.set_xlabel('(mm)')
_ax.set_ylabel('(mm)')
_ax.legend()
plt.show()

In [ ]:
disp((T(e, kx, ky)), ureg.kN)

In [ ]:
disp((C(e, kx, ky)), ureg.kN)

In [ ]:
A_stx(e, kx, ky)

4.2.1 Maximum tension case:

Critical case nr:

In [ ]:
j = i_s
j

Case info:

In [ ]:
LS[j]

In [ ]:
ID[j]

In [ ]:
Side[j]

In [ ]:
Case[j]

In [ ]:
WindDir[j]

Section forces (incl imperfections):

In [ ]:
disp((N[j]), ureg.kN)

In [ ]:
disp((M_x[j]), ureg.kN * ureg.m)

In [ ]:
disp((M_y[j]), ureg.kN * ureg.m)

Resulting strain parameters:

In [ ]:
e, kx, ky = tuple(E[j])
[e, kx, ky]

In [ ]:
NA = Neutral(e, kx, ky)
NA

Concrete strains/stress at corners:

In [ ]:
e_c = epsilon_c(e, kx, ky)
e_c

In [ ]:
disp((vectorize(sigma_c(e_c))), ureg.MPa)

In [ ]:
CGi = CG(e, kx, ky)
CGi

In [ ]:
disp((mc_min(vectorize(sigma_c(e_c)))), ureg.MPa)

Strain/stress in rebar:

In [ ]:
e_s = epsilon_s(e, kx, ky)
e_s

In [ ]:
vectorize(sigma_s(e_s))

In [ ]:
disp((mc_max(vectorize(sigma_s(e_s)))), ureg.MPa)

Utilizations (in terms of strain limits):

In [ ]:
UR_c(e_c)

In [ ]:
UR_s(e_s)

Cross section w/ resulting neutral axis (green dash) for applied forced and location of Compression/Tension resultants (purple stars, lever arm shown dash between resultants):

In [ ]:
disp((z(e, kx, ky)), ureg.mm)

In [ ]:
_fig, _ax = plt.subplots()
_ax.plot(plot_axis(matcol(Contour, 0), ureg.mm), plot_axis(matcol(Contour, 1), ureg.mm), label='matcol(Contour, 0)', color='#00008B')
_ax.plot(plot_axis(X_s, ureg.mm), plot_axis(Y_s, ureg.mm), label='X_s', color='#932329')
_ax.plot(plot_axis(matcol(NA, 0), ureg.mm), plot_axis(matcol(NA, 1), ureg.mm), label='matcol(NA, 0)', color='#068149')
_ax.plot(plot_axis(matcol(CGi, 0), ureg.mm), plot_axis(matcol(CGi, 1), ureg.mm), label='matcol(CGi, 0)', color='#662D91')
_ax.axhline(0, color='0.6', linewidth=0.8)
_ax.axvline(0, color='0.6', linewidth=0.8)
_ax.grid(True, alpha=0.3)
_ax.set_xlabel('(mm)')
_ax.set_ylabel('(mm)')
_ax.legend()
plt.show()

In [ ]:
disp((T(e, kx, ky)), ureg.kN)

In [ ]:
disp((C(e, kx, ky)), ureg.kN)

5 Shear verification

Shear resultant:

In [ ]:
V = vectorize(nth_root(V_x**2 + V_y**2, 2))
disp(V, ureg.kN)

Effective depth

In [ ]:
d = l_c - cov - ø_w - ø / 2
disp(d, ureg.mm)

Shear width

In [ ]:
b_w = w_c

Number of tensile bars:

In [ ]:
n_t = mround(w_c / s_y)
disp(n_t)

Area of reinforcement in tension

In [ ]:
A_st = n_t * ø**2 * (math.pi / 4)
disp(A_st)

Reinforcement ratio:

In [ ]:
rho = mc_min(0.02, A_st / (b_w * d))
disp(rho)

Compression in section:

In [ ]:
sigma_cp = vectorize(-N / A_c)
disp(sigma_cp, ureg.MPa)

Size factor:

In [ ]:
k = mc_min(2, 1 + nth_root(200 * ureg.mm / d, 2))
disp(k)

Shear strenth parameters:

In [ ]:
C_Rdc = 0.18 / 1.5 * ureg.MPa

In [ ]:
k_1 = 0.15

Shear resistance:

In [ ]:
v_Rdc = C_Rdc * k * power(100 * rho * (f_ck / ureg.MPa), 1 / 3)
disp(v_Rdc, ureg.MPa)

In [ ]:
v_min = 0.035 * ureg.MPa * power(k, 3 / 2) * power(f_ck / ureg.MPa, 1 / 2)
disp(v_min, ureg.MPa)

Total design shear resistance:

In [ ]:
V_rd = vectorize((mc_max(v_Rdc, v_min) + k_1 * sigma_cp) * b_w * d)
disp(V_rd, ureg.kN)

Shear utilization

In [ ]:
UR_vc = vectorize(V / V_rd)
disp(UR_vc)

In [ ]:
mc_max(UR_vc)

In [ ]:
print('mc_max(UR_vc) < 1', mc_max(UR_vc) < 1, 'OK!')

6 Minimum reinforcement

Minimum reinforcement for column:

In [ ]:
A_smin = mc_max(vectorize(0.1 * -N) / f_yd, 0.002 * A_c)
disp(A_smin, ureg.mm**2)

Maximum reinforcement:

In [ ]:
A_smax = 0.04 * A_c
disp(A_smax, ureg.mm**2)

Provided vertical reinforcement:

In [ ]:
A_s_total = total(A_s)
disp(A_s_total, ureg.mm**2)

In [ ]:
print('A_s_total > A_smin and A_s_total < A_smax', A_s_total > A_smin and A_s_total < A_smax, 'OK!')

Maximum allowed stirrup spacing:

In [ ]:
s_max = mc_min(400 * ureg.mm, 20 * ø, l_c, w_c)
disp(s_max, ureg.mm)

In [ ]:
print('s_w <= s_max', s_w <= s_max, 'OK!')

7 Bursting at temp bearing

Bearing size:

In [ ]:
l_b = 420 * ureg.mm

In [ ]:
w_b = 490 * ureg.mm

LT91 / 92 -> 650ton -> 420x460mm

LT93 ->300t -> 275x330x

Bearing area

In [ ]:
A_0 = l_b * w_b
disp(A_0, ureg.mm**2)

Max veritical bearing force:

In [ ]:
disp((mc_max(Fz)), ureg.MN)

In [ ]:
d_2 = mc_min(3 * l_b, l_c)
d_2

In [ ]:
b_2 = mc_min(3 * w_b, w_c)
b_2

In [ ]:
A_1 = d_2 * b_2

Confined concrete strength under bearing:

In [ ]:
F_Rdu = A_0 * f_cd * mc_min(nth_root(A_1 / A_0, 2), 3)
disp(F_Rdu, ureg.MN)

Bursting tension force:

In [ ]:
T_burst = 1 / 4 * ((l_c - l_b) / l_c) * mc_max(Fz)
disp(T_burst, ureg.kN)

Increased reinforcement near bearing:

In [ ]:
ø_b = 16 * ureg.mm

In [ ]:
n_leg = 4

Steel area per layer:

In [ ]:
A_swi = n_leg * ø_b**2 * (math.pi / 4)
disp(A_swi, ureg.mm**2)

Height of bursting zone:

In [ ]:
h = l_c
h

Min required stirrups spacing:

In [ ]:
s_req = h * A_swi * f_yd / T_burst
disp(s_req, ureg.mm)

Provided spacing:

In [ ]:
s_b = 300 * ureg.mm

In [ ]:
print('s_b <= s_req', s_b <= s_req, 'OK!')